In [2]:
# GOOGLE DRIVE SPECIFIC FOLDER SCANNER (Ready to Run)
# Target Folder: 170DErJsrMpVN76SlWYvuyNNbJbWAtdQj

from googleapiclient.discovery import build
from google.colab import auth, drive, files
from google.auth import default
from IPython.display import display, HTML

# --- CONFIGURATION ---
# Aapki Folder ID maine yahan set kar di hai
TARGET_FOLDER_ID = "170DErJsrMpVN76SlWYvuyNNbJbWAtdQj"

# Step 1: Authentication
drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
service = build('drive', 'v3', credentials=creds)

output_file = "/content/drive/MyDrive/Target_Folder_PDF_Tree.html"

print(f"Target ID: {TARGET_FOLDER_ID}")
print("Scanning start ho rahi hai... (Badi folders mein time lag sakta hai)")

def get_folder_name(folder_id):
    try:
        file = service.files().get(fileId=folder_id, fields='name', supportsAllDrives=True).execute()
        return file.get('name')
    except:
        return "Unknown Folder"

root_name = get_folder_name(TARGET_FOLDER_ID)
print(f"Folder Name: {root_name}")

def scan_folder_recursive(folder_id):
    """
    Ye function is folder ke andar jitne bhi sub-folders hain,
    un sab ko deep scan karega.
    """
    found_items = []
    # Stack approach use kar rahe hain taaki deep folders mein error na aaye
    folders_to_process = [folder_id]
    processed_count = 0

    while folders_to_process:
        current_id = folders_to_process.pop(0)

        page_token = None
        while True:
            try:
                # Query: Is folder ke bachay dhundo (PDFs ya Folders)
                query = f"'{current_id}' in parents and trashed = false and (mimeType = 'application/vnd.google-apps.folder' or mimeType = 'application/pdf')"

                results = service.files().list(
                    q=query,
                    fields="nextPageToken, files(id, name, mimeType, parents, size)",
                    includeItemsFromAllDrives=True,
                    supportsAllDrives=True,
                    pageSize=1000,
                    pageToken=page_token
                ).execute()

                items = results.get('files', [])

                for item in items:
                    found_items.append(item)
                    # Agar folder hai, to list mein daalo taaki baad mein uske andar bhi check karein
                    if item['mimeType'] == 'application/vnd.google-apps.folder':
                        folders_to_process.append(item['id'])

                page_token = results.get('nextPageToken')
                if not page_token:
                    break
            except Exception as e:
                # Agar koi folder access na ho paye (permission issue), to skip karo
                print(f"Skipping a folder due to error: {e}")
                break

        processed_count += 1
        print(f"Folders Scanned: {processed_count} | Items Found: {len(found_items)}", end='\r')

    return found_items

# --- EXECUTION ---
raw_items = scan_folder_recursive(TARGET_FOLDER_ID)

# --- CLEANING LOGIC (Khali folders hatana) ---
print("\nCleaning empty folders... (Sirf PDFs wale folders rahenge)")

item_map = {item['id']: item for item in raw_items}
relevant_ids = set()

# 1. Sirf PDFs ko pakdo
pdfs = [item for item in raw_items if item['mimeType'] == 'application/pdf']

# 2. Har PDF ka rasta (path) trace karo target folder tak
for pdf in pdfs:
    relevant_ids.add(pdf['id'])
    current_parents = pdf.get('parents', [])

    while current_parents:
        pid = current_parents[0]

        # Agar parent hamare scanned data mein hai
        if pid in item_map:
            if pid in relevant_ids: break # Already added hai to ruko
            relevant_ids.add(pid)
            pid_item = item_map[pid]
            current_parents = pid_item.get('parents', [])
        else:
            # Agar parent scanned data mein nahi hai (matlab hum target folder se upar aa gaye), to stop
            break

# Final filtered list
final_items = [item for item in raw_items if item['id'] in relevant_ids]
print(f"Final Count (PDFs + Active Folders): {len(final_items)}")

# --- HTML GENERATION ---
by_id = {item['id']: item for item in final_items}
children = {}

for item in final_items:
    parents = item.get('parents', [])
    if parents:
        pid = parents[0]
        if pid in by_id:
            children.setdefault(pid, []).append(item)
        elif pid == TARGET_FOLDER_ID:
             # Agar parent wo folder hai jo humne target kiya tha (Root of this scan)
             children.setdefault('target_root', []).append(item)

def sort_kids(kids):
    folders = [x for x in kids if x['mimeType'] == 'application/vnd.google-apps.folder']
    files_only = [x for x in kids if x['mimeType'] != 'application/vnd.google-apps.folder']
    folders.sort(key=lambda x: x['name'].lower())
    files_only.sort(key=lambda x: x['name'].lower())
    return folders + files_only

folder_counter = 0
def new_id():
    global folder_counter
    folder_counter += 1
    return f"f{folder_counter}"

def build_html(item, prefix="", is_last=True):
    name = item['name']
    fid = item['id']
    is_folder = item['mimeType'] == 'application/vnd.google-apps.folder'

    connector = "└── " if is_last else "├── "
    indent = prefix + ("    " if is_last else "│   ")

    size_text = ""
    if not is_folder and 'size' in item:
        s = int(item['size'])
        if s > 1048576: size_text = f" ({s/1048576:.1f} MB)"
        elif s > 0: size_text = f" ({s//1024} KB)"

    if is_folder:
        hid = new_id()
        line = f'{prefix}{connector}<span class="fold" onclick="t(\'{hid}\')">📂 <b>{name}</b></span><br>'
        line += f'<div id="{hid}" class="sub">'
    else:
        view = f"https://drive.google.com/file/d/{fid}/view"
        down = f"https://drive.google.com/uc?id={fid}&export=download"
        line = f'{prefix}{connector}📕 <a href="{view}" target="_blank" class="fn">{name}</a> ' \
               f'<span class="sz">{size_text}</span> ' \
               f'<a href="{down}" download class="dl">⬇</a><br>'

    kids = sort_kids(children.get(fid, []))
    for i, kid in enumerate(kids):
        line += build_html(kid, indent, i == len(kids)-1)

    if is_folder: line += '</div>'
    return line

# Tree Start
tree_html = f'<div class="rootname">📂 {root_name} (PDF Scan)</div>'
root_kids = sort_kids(children.get('target_root', []))

for i, kid in enumerate(root_kids):
    tree_html += build_html(kid, "", i == len(root_kids)-1)

html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>{root_name} - PDF Tree</title>
<style>
  body{{font-family:Segoe UI, sans-serif; padding:20px; background:#f4f6f9}}
  h1{{color:#b71c1c; text-align:center}}
  .rootname{{font-size:24px; font-weight:bold; color:#1a73e8; margin-bottom:20px; border-bottom:2px solid #ddd; padding-bottom:10px}}
  .tree{{background:white; padding:30px; border-radius:12px; box-shadow:0 4px 15px rgba(0,0,0,0.1); font-family:Consolas, monospace; line-height:1.6}}
  .fold{{cursor:pointer; color:#333; font-weight:600}}
  .fold:hover{{color:#1a73e8}}
  .sub{{padding-left:25px; display:block}} /* Default Open */
  .fn{{color:#b71c1c; text-decoration:none; font-weight:500}}
  .fn:hover{{text-decoration:underline}}
  .sz{{color:#777; font-size:0.85em}}
  .dl{{text-decoration:none; margin-left:10px; color:#1a73e8; font-weight:bold}}
  input{{width:100%; padding:15px; margin-bottom:20px; border:2px solid #ddd; border-radius:8px; font-size:16px}}
  .hl{{background:#fff9c4; padding:2px}}
  .info{{text-align:center; margin-bottom:15px; color:#666}}
</style>
</head>
<body>
<h1>PDF Scanner: {root_name}</h1>
<input type="text" placeholder="Search for files in this folder...">
<div class="info">Total Items found: {len(final_items)}</div>
<div class="tree">{tree_html}</div>
<script>
function t(id){{
  var el = document.getElementById(id);
  el.style.display = (el.style.display === 'none') ? 'block' : 'none';
}}
document.querySelector('input').addEventListener('input', function(e){{
    var term = e.target.value.toLowerCase();
    document.querySelectorAll('.fn').forEach(el => {{
        var match = el.innerText.toLowerCase().includes(term);
        el.classList.toggle('hl', match && term !== "");
        if(match && term!=="") {{
            var p = el.parentElement;
            while(p.classList.contains('sub')) {{ p.style.display='block'; p=p.parentElement; }}
        }}
    }});
}});
</script>
</body>
</html>'''

with open(output_file, "w", encoding="utf-8") as f:
    f.write(html)

print(f"\nSUCCESS! Scan mukammal hua.")
print(f"File yahan save hui hai: {output_file}")
files.download(output_file)
display(HTML(html))

Mounted at /content/drive
Target ID: 170DErJsrMpVN76SlWYvuyNNbJbWAtdQj
Scanning start ho rahi hai... (Badi folders mein time lag sakta hai)
Folder Name: Year 2025
Folders Scanned: 11 | Items Found: 76
Cleaning empty folders... (Sirf PDFs wale folders rahenge)
Final Count (PDFs + Active Folders): 76

SUCCESS! Scan mukammal hua.
File yahan save hui hai: /content/drive/MyDrive/Target_Folder_PDF_Tree.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>